In [ ]:
print("hello world!")

In [ ]:
import json
from pathlib import Path

from xdomain_ser.core import data_helper as dh, ser
from xdomain_ser.cli import (
    _load_extractor,
    _extract_one,
    _translate_lora_mr,
    _ser_report,
    _resolve_checkpoint_dir,
    _resolve_hint_map_path,
    _resolve_prompt_examples_path,
)

## Load the LoRA extractor

We use the private CLI helpers directly. They cache the model in a process-level dict so subsequent calls don't reload.

In [ ]:
checkpoint = _resolve_checkpoint_dir(None)
hint_map_path = _resolve_hint_map_path(None)
prompt_examples_path = _resolve_prompt_examples_path(None)

print(f"checkpoint:        {checkpoint}")
print(f"hint_map_path:     {hint_map_path}")
print(f"prompt_examples:   {prompt_examples_path}")

model, tokenizer, hint_map_table, pool = _load_extractor(
    checkpoint, hint_map_path, prompt_examples_path
)
print("\nLoaded. Available domains:", sorted(hint_map_table.keys())[:5], "...")

## Score a single example

Pick an E2E restaurant description and a gold MR. We do the extraction, then translate the LoRA's internal slot names back to E2E-native names, then compute the SER + slot-F1 report.

In [ ]:
text = (
    "Cotto is a coffee shop near The Portland Arms that serves "
    "English food at a high price with average customer rating."
)
gold = {
    "name": "Cotto",
    "eatType": "coffee shop",
    "food": "English",
    "priceRange": "high",
    "customerRating": "average",
    "near": "The Portland Arms",
}

pred_str = _extract_one(text, "hm_e2e_nlg", model, tokenizer, hint_map_table, pool)
pred_dict = dict(ser.extract_attributes_dict(pred_str.replace(dh.LIST_END, "")))
pred_native = _translate_lora_mr(pred_dict, "e2e")
report = _ser_report(pred_native, gold)

print(f"Predicted (LoRA-internal): {pred_dict}")
print(f"Predicted (E2E-native):    {pred_native}")
print(f"Gold:                      {gold}")
print()
print(f"SER:      {report['SER']:.4f}  (S={report['S']}, D={report['D']}, I={report['I']}, N_ref={report['N_ref']})")
print(f"Slot F1:  {report['slot_f1']:.4f}  (P={report['slot_precision']:.4f}, R={report['slot_recall']:.4f})")

## Score a batch from the shipped gold-annotated set

We use the first 20 LLM-sourced examples from `evaluation/gold/gold-annotated.json` to demonstrate batch scoring. Each example already has the `mr` (reference) and `clean_pred_text` (the M2T model's output).

In [ ]:
#import os
#print(os.getcwd())

gold_path = Path("../evaluation/gold/gold-annotated.json")
with gold_path.open() as f:
    gold_data = json.load(f)

llm_examples = [ex for ex in gold_data if ex["source"] == "llm"][:20]
print(f"Loaded {len(llm_examples)} LLM-sourced examples")

In [ ]:
results = []
for ex in llm_examples:
    text = ex["clean_pred_text"]
    gold = ex["mr"]

    pred_str = _extract_one(text, "hm_e2e_nlg", model, tokenizer, hint_map_table, pool)
    pred_dict = dict(ser.extract_attributes_dict(pred_str.replace(dh.LIST_END, "")))
    pred_native = _translate_lora_mr(pred_dict, "e2e")
    report = _ser_report(pred_native, gold)

    results.append({
        "id": ex["id"],
        "personality": ex["personality"],
        "SER": report["SER"],
        "slot_f1": report["slot_f1"],
    })

for r in results:
    print(f"  {r['id']:>20s}  pers={r['personality']:<22s}  SER={r['SER']:.3f}  F1={r['slot_f1']:.3f}")

In [ ]:
import statistics

mean_ser = statistics.mean(r["SER"] for r in results)
mean_f1 = statistics.mean(r["slot_f1"] for r in results)
print(f"\nMean SER over {len(results)} examples: {mean_ser:.4f}")
print(f"Mean slot F1:                   {mean_f1:.4f}")

## Next steps

* See `02_new_domain_hint_map.ipynb` to score outputs from a domain not in the shipped hint map.
* See `03_inspect_routing.ipynb` to visualize the LoRA-vs-NLI routing decisions.
* For batch scoring from the CLI, use `xdomain-ser score --input-file <json>` (documented in [USAGE.md](../USAGE.md)).